# Basic Math and Arithmetic Functions
Applying built-in Spark math functions to DataFrame columns for numeric computation and transformation.

### Links and Resources
- [abs()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.abs.html)
- [round()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.round.html)
- [ceil() / floor()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.ceil.html)
- [sqrt()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.sqrt.html)
- [pow()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.pow.html)
- [log()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.log.html)
- [PySpark SQL Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

📌 Spark session setup

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

📌 Reading data into a Spark DataFrame

In [ ]:
df = spark.read.table("population_metrics.default.countries_population")
display(df)

In [ ]:
from pyspark.sql.functions import col, round, format_number

result = df.withColumn("population_forcast_2030", format_number(round(col("POPULATION") * 1.2), 0))
display(result)

📌 Basic arithmetic — operators on columns

In [ ]:
from pyspark.sql.functions import col, round

df.selectExpr(
    "NAME AS Country",
    "POPULATION",
    "AREA_KM2",
    "POPULATION + 1000          AS Pop_Plus_1000",
    "POPULATION * 2             AS Pop_Doubled",
    "POPULATION / AREA_KM2      AS Pop_Per_Km2_Raw"
).show(5)

📌 round(), ceil(), floor() — controlling decimal precision

In [ ]:
from pyspark.sql.functions import ceil, floor

df.withColumn("Density_Raw",   col("POPULATION") / col("AREA_KM2")) \
  .withColumn("Density_Round", round(col("Density_Raw"), 2)) \
  .withColumn("Density_Ceil",  ceil(col("Density_Raw"))) \
  .withColumn("Density_Floor", floor(col("Density_Raw"))) \
  .select("NAME", "Density_Raw", "Density_Round", "Density_Ceil", "Density_Floor") \
  .show(5)

📌 abs() — absolute value

In [ ]:
from pyspark.sql.functions import abs, lit

# Simulate a signed delta and take its absolute value
df.withColumn("Pop_Delta",     col("POPULATION") - lit(10_000_000)) \
  .withColumn("Pop_Delta_Abs", abs(col("Pop_Delta"))) \
  .select("NAME", "POPULATION", "Pop_Delta", "Pop_Delta_Abs") \
  .show(5)

📌 sqrt() and pow() — square root and exponentiation

In [ ]:
from pyspark.sql.functions import sqrt, pow

df.withColumn("Sqrt_Area",   round(sqrt(col("AREA_KM2")), 2)) \
  .withColumn("Area_Squared", pow(col("AREA_KM2"), 2)) \
  .select("NAME", "AREA_KM2", "Sqrt_Area", "Area_Squared") \
  .show(5)

📌 log() — natural log and log base 10

In [ ]:
from pyspark.sql.functions import log, log10

df.withColumn("Log_Population",   round(log(col("POPULATION")), 4)) \
  .withColumn("Log10_Population", round(log10(col("POPULATION")), 4)) \
  .select("NAME", "POPULATION", "Log_Population", "Log10_Population") \
  .show(5)

---
## 📅 Date Functions
Common Spark date functions for working with dates, timestamps, and date arithmetic.

### Links and Resources
- [current_date() / current_timestamp()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.current_date.html)
- [date_add() / date_sub()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_add.html)
- [datediff()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.datediff.html)
- [year() / month() / dayofmonth()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.year.html)
- [date_format()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.date_format.html)
- [to_date()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.to_date.html)

📌 current_date() / current_timestamp() — today's date and time

In [ ]:
from pyspark.sql.functions import current_date, current_timestamp, lit

spark.range(1).select(
    current_date().alias("Today"),
    current_timestamp().alias("Now")
).show(truncate=False)

📌 date_add() / date_sub() / datediff() — date arithmetic

In [ ]:
from pyspark.sql.functions import current_date, date_add, date_sub, datediff, to_date, lit

forecast_target = to_date(lit("2030-01-01"))

spark.range(1).select(
    current_date().alias("Today"),
    date_add(current_date(), 30).alias("Plus_30_Days"),
    date_sub(current_date(), 30).alias("Minus_30_Days"),
    datediff(forecast_target, current_date()).alias("Days_Until_2030")
).show(truncate=False)

📌 year() / month() / dayofmonth() / dayofweek() — extracting date parts

In [ ]:
from pyspark.sql.functions import current_date, year, month, dayofmonth, dayofweek

spark.range(1).select(
    current_date().alias("Date"),
    year(current_date()).alias("Year"),
    month(current_date()).alias("Month"),
    dayofmonth(current_date()).alias("Day"),
    dayofweek(current_date()).alias("DayOfWeek")  # 1=Sunday, 7=Saturday
).show(truncate=False)

📌 date_format() — custom date formatting

In [ ]:
from pyspark.sql.functions import current_date, date_format

spark.range(1).select(
    current_date().alias("Date_Raw"),
    date_format(current_date(), "dd/MM/yyyy").alias("EU_Format"),
    date_format(current_date(), "MMMM dd, yyyy").alias("Long_Format"),
    date_format(current_date(), "yyyy-'Q'Q").alias("Quarter"),
    date_format(current_date(), "EEEE").alias("Weekday_Name")
).show(truncate=False)

📌 date_diff_ymd() — custom UDF: human-readable date difference (e.g. `1 y 2 m 3 d`)

In [ ]:
from pyspark.sql.functions import (
    floor, months_between, add_months, datediff,
    when, concat, lit, rtrim,
)
from pyspark.sql.types import IntegerType


def date_diff_ymd(start_col, end_col):
    """
    Returns a Spark Column expression with the calendar difference between
    two date columns as a human-readable string, e.g. '1 y 2 m 3 d'.

    Uses only built-in Spark functions — no Python UDF — so it works on
    Databricks Serverless regardless of the server's Python version.

    Leap years are handled naturally: add_months() clips to the last valid
    day of the month (e.g. Jan 30 + 1 month → Feb 29 in a leap year),
    and datediff() counts the exact remaining days from that point.
    """
    # Always compute in the forward direction
    s = when(start_col <= end_col, start_col).otherwise(end_col)
    e = when(start_col <= end_col, end_col).otherwise(start_col)

    # months_between returns a float; floor gives complete calendar months
    full_months = floor(months_between(e, s)).cast(IntegerType())
    yrs  = (full_months / 12).cast(IntegerType())
    mths = full_months % 12

    # Advance start by all complete months, then count leftover days
    dys = datediff(e, add_months(s, full_months))

    # Build string components — days always shown when the total difference is zero
    y_str = when(yrs  > 0,                       concat(yrs .cast("string"), lit(" y "))).otherwise(lit(""))
    m_str = when(mths > 0,                       concat(mths.cast("string"), lit(" m "))).otherwise(lit(""))
    d_str = when((dys > 0) | (full_months == 0), concat(dys .cast("string"), lit(" d")) ).otherwise(lit(""))

    result = rtrim(concat(y_str, m_str, d_str))

    return when(
        start_col.isNull() | end_col.isNull(),
        lit(None).cast("string")
    ).otherwise(result)

In [ ]:
from pyspark.sql.functions import col, to_date

examples = spark.createDataFrame([
    ("Exact year",           "2023-03-04", "2024-03-04"),  # → 1 y
    ("Leap day as start",    "2024-02-29", "2025-03-01"),  # add_months clips: +12 m → 2025-02-28, +1 d → 1 y 1 d
    ("Across Feb 29 (leap)", "2024-01-30", "2024-03-01"),  # +1 m → 2024-02-29, +2 d → 1 m 2 d
    ("Across Feb 28",        "2023-01-30", "2023-03-01"),  # +1 m → 2023-02-28, +1 d → 1 m 1 d
    ("Mixed y/m/d",          "2021-11-15", "2024-03-04"),  # → 2 y 3 m 18 d
    ("Same day",             "2024-06-10", "2024-06-10"),  # → 0 d
    ("NULL end",             "2024-01-01", None),           # → null
], ["Label", "Start", "End"])

examples = (
    examples
    .withColumn("Start", to_date("Start"))
    .withColumn("End",   to_date("End"))
    .withColumn("Difference", date_diff_ymd(col("Start"), col("End")))
)

display(examples)

## 📌 String Functions
Applying built-in Spark string functions to DataFrame columns for text manipulation and analysis.

### Links and Resources
- [length()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.length.html)
- [upper() / lower()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.upper.html)
- [trim() / ltrim() / rtrim()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.trim.html)
- [substring()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.substring.html)
- [concat() / concat_ws()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.concat.html)
- [replace()](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/api/pyspark.sql.functions.regexp_replace.html)
- [PySpark SQL Functions](https://spark.apache.org/docs/latest/api/python/reference/pyspark.sql/functions.html)

In [ ]:
from pyspark.sql.functions import length, lit

df.select(
    col("NAME"),
    lit("Hello Sample").alias("Sample_Text"),
    length(col("NAME")).alias("Name_Length")
)